<a href="https://colab.research.google.com/github/Py-saqlain/Movie_Recap/blob/main/Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**MOVIE RECAP ENGINE | DAY 01**

In [ ]:
!apt-get install -y ffmpeg -q
!pip install openai-whisper faiss-cpu sentence-transformers -q
print("Done. No errors above = we are good.")

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

# Create our project folder in Drive
PROJECT_DIR = "/content/drive/MyDrive/recap_engine"
os.makedirs(PROJECT_DIR, exist_ok=True)

print(f"Drive mounted. Project folder ready at: {PROJECT_DIR}")

In [ ]:
import subprocess, json

VIDEO = "/content/drive/MyDrive/MovieApp/movie.mp4"

out = subprocess.run(
    ["ffprobe", "-v", "quiet", "-print_format", "json",
     "-show_format", "-show_streams", VIDEO],
    capture_output=True, text=True
)
info = json.loads(out.stdout)

for s in info["streams"]:
    print(f"{s['codec_type'].upper()} → codec: {s['codec_name']} | duration: {s.get('duration','?')}s")

total = float(info["format"]["duration"])
print(f"\nTotal duration : {total:.1f}s = {total/60:.1f} minutes")
print(f"We will start  : at 20s (skip intro)")
print(f"We will stop   : at {total-60:.1f}s (skip outro)")

In [ ]:
import whisper, subprocess

TEST_CLIP = "/content/test_clip.mp4"

# Cut 60s from 2 min mark — stream copy, no re-encode
subprocess.run([
    "ffmpeg", "-y",
    "-ss", "120",
    "-i", VIDEO,
    "-t", "60",
    "-c", "copy",
    TEST_CLIP
], check=True)
print("Clip cut successfully.\n")

# Run Whisper on that 60s clip
model = whisper.load_model("base")
result = model.transcribe(TEST_CLIP, word_timestamps=True)

print(f"Language detected: {result['language']}\n")
print("First 5 segments:")
for seg in result["segments"][:5]:
    print(f"  [{seg['start']:.2f}s → {seg['end']:.2f}s]  {seg['text'].strip()}")

In [ ]:
import faiss, numpy as np, shutil, os
from sentence_transformers import SentenceTransformer

# Test embeddings
embedder = SentenceTransformer("all-MiniLM-L6-v2")
sentences = [
    "The hero fights the villain in the rain",
    "She discovers a hidden secret in the basement",
    "They kiss goodbye at the train station",
]
vecs = embedder.encode(sentences).astype("float32")

# Test FAISS
index = faiss.IndexFlatL2(vecs.shape[1])
index.add(vecs)
query = embedder.encode(["action fight scene"]).astype("float32")
D, I = index.search(query, k=2)

print("FAISS search test:")
print(f"  Query: 'action fight scene'")
for rank, i in enumerate(I[0]):
    print(f"  Rank {rank+1}: {sentences[i]}")

# Save test clip to Drive
DRIVE_DIR = "/content/drive/MyDrive/MovieApp"
shutil.copy(TEST_CLIP, f"{DRIVE_DIR}/day1_test_clip.mp4")

print(f"\nCheckpoint saved to Drive.")
print("\n Day 1 COMPLETE.")
print(f"Movie : Evil Returns / movie.mp4 — 115.6 min")
print(f"Codec : H264 + AAC — perfect format")
print(f"Whisper : working with timestamps")
print(f"FAISS : working with embeddings")
print(f"Tomorrow : Day 2 — we cut real clips from this movie")

In [ ]:
import os

files = os.listdir("/content/drive/MyDrive/MovieApp")
print("Files in your MovieApp folder:")
for f in files:
    size = os.path.getsize(f"/content/drive/MyDrive/MovieApp/{f}")
    print(f"  {f}  →  {size/1_000_000:.1f} MB")

**DAY 02 ( 4 cells  )**

In [ ]:
!apt-get install -y ffmpeg -q
!pip install openai-whisper faiss-cpu sentence-transformers -q

from google.colab import drive
import os, subprocess, json, shutil

drive.mount("/content/drive")

VIDEO     = "/content/drive/MyDrive/MovieApp/movie.mp4"
DRIVE_DIR = "/content/drive/MyDrive/MovieApp"

print(f"File exists: {os.path.exists(VIDEO)}")
print("Ready for Day 2.")

In [ ]:
os.makedirs("/content/clips", exist_ok=True)

SCENES = [
    ("scene_01", 120,  180),
    ("scene_02", 600,  680),
    ("scene_03", 2400, 2480),
    ("scene_04", 5400, 5480),
]

clip_paths = []

for name, start, end in SCENES:
    out = f"/content/clips/{name}.mp4"
    duration = end - start

    subprocess.run([
        "ffmpeg", "-y",
        "-ss", str(start),
        "-i", VIDEO,
        "-t", str(duration),
        "-c:v", "libx264",    # re-encode video — forces exact frame alignment
        "-c:a", "aac",        # re-encode audio
        "-crf", "18",         # quality level — 0 best, 51 worst, 18 = near lossless
        "-preset", "ultrafast", # fastest encoding speed
        "-avoid_negative_ts", "make_zero",  # fixes timestamp issues
        out
    ], check=True)

    size = os.path.getsize(out) / 1_000_000
    clip_paths.append(out)
    print(f"{name} → {start}s to {end}s → {size:.1f} MB ✔")

print(f"\nAll 4 clips cut. Note: re-encoding takes longer than stream copy.")

In [ ]:
concat_file = "/content/concat_list.txt"
merged      = "/content/day2_merged_fixed.mp4"

with open(concat_file, "w") as f:
    for path in clip_paths:
        f.write(f"file '{path}'\n")

subprocess.run([
    "ffmpeg", "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", concat_file,
    "-c:v", "libx264",      # re-encode final merge too
    "-c:a", "aac",
    "-crf", "18",
    "-preset", "ultrafast",
    merged
], check=True)

size = os.path.getsize(merged) / 1_000_000
print(f"Fixed merged video → {size:.1f} MB")

In [ ]:
probe = subprocess.run(
    ["ffprobe", "-v", "quiet", "-print_format", "json",
     "-show_format", "-show_streams", merged],
    capture_output=True, text=True
)
info = json.loads(probe.stdout)

print("Stream check:")
for s in info["streams"]:
    print(f"  {s['codec_type'].upper()} → {s.get('duration','?')}s")

actual   = float(info["format"]["duration"])
expected = sum(end - start for _, start, end in SCENES)

print(f"\nExpected : {expected}s")
print(f"Actual   : {actual:.2f}s")
print(f"Diff     : {abs(actual - expected):.2f}s")

if abs(actual - expected) < 2:
    print("SYNC CHECK PASSED ✔")

shutil.copy(merged, f"{DRIVE_DIR}/day2_merged_fixed.mp4")
print(f"Saved → day2_merged_fixed.mp4")